In [1]:
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dataset = load_dataset("stanfordnlp/imdb", cache_dir="../data/imdb")
print(dataset)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
print("Tokenizer loaded")

Using device: cuda


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Tokenizer loaded


In [3]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=512
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(tokenized)
print(f"\nSample input_ids shape: {tokenized['train'][0]['input_ids'].shape}")
print(f"Sample attention_mask shape: {tokenized['train'][0]['attention_mask'].shape}")

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

Sample input_ids shape: torch.Size([512])
Sample attention_mask shape: torch.Size([512])


In [4]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score
import numpy as np

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model = model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

args = TrainingArguments(
    output_dir='./results/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

print("Model and trainer ready")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Model and trainer ready
Model parameters: 66,955,010


In [5]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.269595,0.301658,0.902800
2,0.192051,0.278776,0.928600
3,0.114296,0.312574,0.934200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9375, training_loss=0.20736108764648437, metrics={'train_runtime': 11216.7279, 'train_samples_per_second': 6.686, 'train_steps_per_second': 0.836, 'total_flos': 9935054899200000.0, 'train_loss': 0.20736108764648437, 'epoch': 3.0})

In [6]:
model.save_pretrained('./results/baseline_model')
tokenizer.save_pretrained('./results/baseline_model')
print("Model saved to ./results/baseline_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./results/baseline_model


In [9]:
import shutil
import os

src = './results/checkpoints'
dst = '../results/checkpoints'

shutil.copytree(src, dst)
shutil.rmtree(src)

print(f"Moved to: {os.path.abspath(dst)}")

Moved to: c:\Users\admin\Documents\Projects\transformer-uncertainity\results\checkpoints
